In [1]:
import torch
import torchvision.models as models
from torch.profiler import profile, ProfilerActivity

model = models.resnet18()
inputs = torch.randn(5, 3, 224, 224)

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    model(inputs)

ERROR:2026-01-09 15:17:34 76724:76724 DeviceProperties.cpp:47] gpuGetDeviceCount failed with code 35


In [6]:
print(prof.key_averages().table(sort_by="cpu_time_total",row_limit=10))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     aten::conv2d         0.34%     929.933us        82.48%     228.760ms      11.438ms            20  
                aten::convolution         0.44%       1.212ms        82.14%     227.830ms      11.392ms            20  
               aten::_convolution         0.31%     848.953us        81.70%     226.619ms      11.331ms            20  
         aten::mkldnn_convolution        80.82%     224.159ms        81.40%     225.770ms      11.288ms            20  
                 aten::batch_norm         0.13%     360.517us         8.24%      22.852ms       1.143ms            20  
     aten::_batch_norm_impl_index       

In [8]:
print(prof.key_averages(group_by_input_shape=True).table(sort_by="cpu_time_total",row_limit=30))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls                                                                      Input Shapes  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                     aten::conv2d         0.03%      88.864us        22.29%      61.817ms      15.454ms             4                             [[5, 64, 56, 56], [64, 64, 3, 3], [], [], [], [], []]  
                aten::convolution         0.08%     216.933us        22.25%      61.728ms      15.432ms             4                     [[5, 64, 56, 56], [64, 64, 3, 3], [], [], [], [], [], 

In [9]:
prof.export_chrome_trace("trace.json")

In [11]:
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    for i in range(10):
        model(inputs)
        prof.step()
prof.export_chrome_trace("trace2.json")

In [12]:
print(prof.key_averages(group_by_input_shape=True).table(sort_by="cpu_time_total",row_limit=30))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls                                                                      Input Shapes  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                     aten::conv2d         0.05%     777.249us        18.58%     305.744ms       7.644ms            40                             [[5, 64, 56, 56], [64, 64, 3, 3], [], [], [], [], []]  
                aten::convolution         0.12%       2.038ms        18.54%     304.967ms       7.624ms            40                     [[5, 64, 56, 56], [64, 64, 3, 3], [], [], [], [], [], 

In [13]:
from torch.profiler import profile, tensorboard_trace_handler
with profile(activities=[ProfilerActivity.CPU], record_shapes=True, on_trace_ready=tensorboard_trace_handler("./log/resnet18")) as prof:
    for i in range(10):
        model(inputs)
        prof.step()

[W109 15:33:21.236406984 collection.cpp:663] Warning: aten::resolve_conj (function operator())
